#  Arm Assembly Primer : Loops
# 1. A loop is a branch going back
* A loop repeats instructions by jumping _back_ to an earlier label


In [ ]:
%%writefile loop.c
long sum(long * a, long n) {
  long s = 0;
  for (long i = 0; i < n; i++) {
    s += a[i];
  }
  return s;
}

void fill(long * a, long n, long v) {
  for (long i = 0; i < n; i++) {
    a[i] = v;
  }
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang -O1 -S loop.c
grep -vE '^\s+\.|^\s*//|^\.Lfunc' loop.s

* You will see something like this for `sum` (`a` is in `x0`, `n` in `x1`)

```
sum:
	cmp	x1, #1
	b.lt	.LBB0_4            // if n < 1, skip the loop
	mov	x8, xzr            // s = 0
.LBB0_2:                         // the loop begins here
	ldr	x9, [x0], #8       // x9 = *a; then a = a + 8
	subs	x1, x1, #1         // n = n - 1 (and set the flags)
	add	x8, x9, x8         // s = s + x9
	b.ne	.LBB0_2            // if n != 0, go back to .LBB0_2
	mov	x0, x8             // return s
	ret
.LBB0_4:
	mov	x8, xzr
	mov	x0, x8             // return 0
	ret
```

* `b.ne .LBB0_2` jumps back to the beginning of the loop; this is what makes it a loop
* The compiler rewrote the loop quite a bit
  * there is no `i`; instead, the pointer `a` itself advances by 8 bytes (one `long`) every iteration. `ldr x9, [x0], #8` loads from the address in `x0` and then adds 8 to `x0` (_post-increment_)
  * `n` counts down to zero (`subs x1, x1, #1`), so the end of the loop can be detected by the flags set by `subs` (`subs` is `sub` that also sets the flags), without a separate `cmp`
  * the check whether the loop runs at all (`n < 1`) is done once, before the loop
* So <font color="blue">the assembly code of a loop rarely looks like the source</font>; look for a label and a branch going back to it, then see what is done in between
* The compiler marks the beginning of a loop with a comment `// =>This Inner Loop Header: Depth=1`, which `grep` above did not hide
* Visualizing shows the loop as a cycle


In [ ]:
import vis_asm
vis_asm.vis_asm_file("loop.s")

* `fill` is similar; `str x2, [x0], #8` stores `v` at the address in `x0`, then adds 8 to `x0`

# 2. `while` loops
* A `while` loop is no different; here is a loop whose number of iterations is not known in advance


In [ ]:
%%writefile while.c
/* the number of steps for x to reach 1 by the Collatz rule */
long count_steps(long x) {
  long steps = 0;
  while (x != 1) {
    if (x % 2 == 0) {
      x = x / 2;
    } else {
      x = 3 * x + 1;
    }
    steps++;
  }
  return steps;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang -O1 -S while.c
grep -vE '^\s+\.|^\s*//|^\.Lfunc' while.s

* You will see something like this

```
count_steps:
	cmp	x0, #1
	b.ne	.LBB0_2            // if x != 1, enter the loop
	mov	x0, xzr            // return 0
	ret
.LBB0_2:
	mov	x8, xzr            // steps = 0
.LBB0_3:                         // the loop begins here
	asr	x9, x0, #1         // x9 = x / 2  (shift right by 1 bit)
	add	x10, x0, x0, lsl #1  // x10 = x + 2 * x = 3 * x
	tst	x0, #0x1           // test the lowest bit of x (is x odd?)
	add	x8, x8, #1         // steps++
	csinc	x0, x9, x10, eq    // x = (x is even) ? x9 : x10 + 1
	cmp	x0, #1
	b.ne	.LBB0_3            // if x != 1, go back
	mov	x0, x8             // return steps
	ret
```

* The picture shows the loop as a cycle (the block from `.LBB0_3` to `b.ne .LBB0_3` jumping back to itself), and the check before the loop as a branch that either enters the loop or returns 0 at once


In [ ]:
import vis_asm
vis_asm.vis_asm_file("while.s")

* The `if` statement inside the loop has become a conditional select (`csinc` = "conditional select and increment": `x9` if the condition holds, `x10 + 1` otherwise); both `x / 2` and `3 * x` are computed every iteration
* `x % 2 == 0` became a test of the lowest bit (`tst x0, #0x1`), and `x / 2` a shift (`asr`), both cheaper than a division

# 3. A note on optimization levels
* We have compiled with `-O1`. With `-O2` or `-O3`, the compiler makes `sum` and `fill` process several elements per instruction using _SIMD instructions_, and their assembly code becomes much longer. This is the topic of pd07_simd
* With `-O0`, the compiler keeps `s`, `i`, `a` and `n` in memory and loads/stores them every time, which is slow and verbose. Try it if you are curious

# 4. Summary
* a loop becomes a sequence of instructions ending with a (conditional) branch back to its beginning
* the compiler transforms loops a lot (pointers advancing, counting down, checks moved out of the loop, `if` turned into `csel`); look for a label and a branch back to it